# 🌲 Scikit-learn Baseline Model & Feature Comparison

This notebook trains baseline `DecisionTreeClassifier` models using standard scikit-learn. It compares performance across different feature representations:
1. **Raw Flattened Signals**: Raw accelerometer time steps concatenated.
2. **Handcrafted Statistical Features**: Mean, std, min, max, median, energy, zero-crossing rate, IQR.
3. **TSFEL Features (Optional)**: Automatically extracted features using the Time Series Feature Extraction Library (TSFEL).

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Ensure project root is in path for imports
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils.helpers import load_config, set_seed
from scripts.train import extract_statistical_features

config = load_config("config/config.yaml")
seed = config["training"]["random_seed"]
set_seed(seed)

sns.set_theme(style="whitegrid")

## 📂 Dataset Preparation

We attempt to load the preprocessed HAR dataset. If the raw UCI HAR Dataset is missing, we automatically fall back to generating synthetic accelerometer windows representing 6 activities.

In [ ]:
has_real_data = False
try:
    from src.data.preprocessing import load_and_split_dataset
    X_train, X_test, y_train, y_test = load_and_split_dataset(config)
    # Standard window size is 500 samples, 3 axes
    X_all = np.concatenate([X_train, X_test])
    y_all = np.concatenate([y_train, y_test])
    has_real_data = True
    print(f"✅ Successfully loaded real dataset: X shape = {X_all.shape}, y shape = {y_all.shape}")
except Exception:
    print("⚠️ Dataset not found. Generating synthetic accelerometer window dataset...")
    # Generate synthetic 3D window data: 300 samples, 500 length, 3 channels, 6 classes
    np.random.seed(seed)
    n_samples = 300
    seq_len = 500
    n_channels = 3
    
    X_all = np.random.randn(n_samples, seq_len, n_channels)
    # Add periodic signals or offsets depending on the class label to make it learnable
    y_all = np.random.choice([1, 2, 3, 4, 5, 6], size=n_samples)
    for i in range(n_samples):
        label = y_all[i]
        if label <= 3: # dynamic activities: high frequency waves
            X_all[i, :, 0] += np.sin(np.linspace(0, 10 * label, seq_len))
        else: # static activities: constant offset (gravity)
            X_all[i, :, 1] += 0.5 * label
            
    print(f"✅ Generated synthetic dataset: X shape = {X_all.shape}, y shape = {y_all.shape}")

## 📐 Feature Extraction Comparison

We create three different feature matrices:
1. **Raw Flattened**: Shape `(n_samples, 500 * 3 = 1500)`
2. **Handcrafted Statistical**: Shape `(n_samples, 3 * 8 = 24)`

In [ ]:
# 1. Raw flattened
X_raw = X_all.reshape(X_all.shape[0], -1)

# 2. Handcrafted statistical features
X_statistical = extract_statistical_features(X_all)

print(f"Raw features shape        : {X_raw.shape}")
print(f"Statistical features shape: {X_statistical.shape}")

## 🌲 Training baseline models

We split the datasets and evaluate decision trees trained on raw signals vs statistical features.

In [ ]:
def evaluate_features(X, y, title):
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed, stratify=y)
    
    # Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # Train
    clf = DecisionTreeClassifier(max_depth=5, random_state=seed)
    clf.fit(X_train_scaled, y_train)
    
    # Predict & Evaluate
    y_pred = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"=== {title} Accuracy: {acc:.4f} ===")
    print(classification_report(y_test, y_pred, zero_division=0))
    return acc

acc_raw = evaluate_features(X_raw, y_all, "Raw Flattened Signals")
acc_stat = evaluate_features(X_statistical, y_all, "Handcrafted Statistical Features")

## 📈 Tree Depth vs Accuracy

We plot tree depth against test set accuracy to observe overfitting behavior on raw vs engineered features.

In [ ]:
depths = list(range(2, 16))
accuracies_raw = []
accuracies_stat = []

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, y_all, test_size=0.3, random_state=seed, stratify=y_all)
X_train_stat, X_test_stat, _, _ = train_test_split(X_statistical, y_all, test_size=0.3, random_state=seed, stratify=y_all)

scaler_raw = StandardScaler()
X_train_raw_s = scaler_raw.fit_transform(X_train_raw)
X_test_raw_s = scaler_raw.transform(X_test_raw)

scaler_stat = StandardScaler()
X_train_stat_s = scaler_stat.fit_transform(X_train_stat)
X_test_stat_s = scaler_stat.transform(X_test_stat)

for depth in depths:
    # Raw signals model
    clf_raw = DecisionTreeClassifier(max_depth=depth, random_state=seed)
    clf_raw.fit(X_train_raw_s, y_train)
    accuracies_raw.append(accuracy_score(y_test, clf_raw.predict(X_test_raw_s)))
    
    # Statistical features model
    clf_stat = DecisionTreeClassifier(max_depth=depth, random_state=seed)
    clf_stat.fit(X_train_stat_s, y_train)
    accuracies_stat.append(accuracy_score(y_test, clf_stat.predict(X_test_stat_s)))

plt.figure(figsize=(10, 6))
plt.plot(depths, accuracies_raw, marker='o', label='Raw Accelerometer', color="red")
plt.plot(depths, accuracies_stat, marker='s', label='Statistical Features', color="green")
plt.xlabel("Tree Depth")
plt.ylabel("Test Accuracy")
plt.title("Decision Tree Accuracy vs Depth")
plt.xticks(depths)
plt.grid(True)
plt.legend()
plt.show()